# VerinBirds — Этап 4: CNN transfer learning БЕЗ GPU (linear probe, Colab CPU)

Этот ноутбук **не требует GPU** и работает на бесплатном Colab даже без compute units.
Идея — *linear probe* (признанный вид transfer learning):
1. Берём предобученный **ResNet18** как **замороженный экстрактор признаков** (только forward-проход,
   свёртки НЕ обучаем, backprop нет).
2. Прогоняем фото → получаем 512-мерные эмбеддинги.
3. Поверх обучаем **логистическую регрессию** (доли секунды).

Почему так:
- GPU не нужен, телефон-верификация не нужна, всё крутится на серверах Colab (твой ноут не греется).
- Forward-проход на подвыборке — это ~5-10 мин на CPU, а не полчаса backprop.
- Это честный transfer learning для конкурса, и тем же экстрактором мы прогоним перепелов (Этап 6).

**Честная рамка:** высокая точность на курином test ожидаема (классы разделимы по цвету/источнику,
baseline дал 99.8%). Реальная проверка — перенос на перепелов.

## Что сделать
1. Runtime → тип можно оставить CPU (GPU не нужен).
2. Ячейка «Данные»: смонтируй Drive и укажи путь к `kaggle_archive.zip` (тот, что ты делал).
3. Runtime → Run all. В конце файлы сохранятся в Drive (`verinbirds_out/`) и метрики напечатаются — сохрани их.


In [ ]:
# Ячейка 1 — окружение (GPU не обязателен)
import torch, torchvision, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "(CPU — это ок для linear probe)")


In [ ]:
# Ячейка «Данные» — Google Drive
from google.colab import drive
drive.mount('/content/drive')
import zipfile, os, glob
# >>> ПОПРАВЬ путь, если архив в другом месте Drive:
DRIVE_ARCHIVE = '/content/drive/MyDrive/kaggle_archive.zip'
DEST = '/content/data/chicken'
os.makedirs(DEST, exist_ok=True)
if not glob.glob(DEST + '/**/*.jpg', recursive=True):
    with zipfile.ZipFile(DRIVE_ARCHIVE) as z: z.extractall(DEST)
imgs = glob.glob(DEST + '/**/*.jpg', recursive=True) + glob.glob(DEST + '/**/*.png', recursive=True)
print("изображений:", len(imgs))
assert len(imgs) > 3000, "Проверь путь DRIVE_ARCHIVE — архив не найден/битый"


In [ ]:
# Ячейка 2 — препроцессинг (letterbox) + сбор файлов, ПОДВЫБОРКА train для скорости
import random, numpy as np
from pathlib import Path
from PIL import Image, ImageOps
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
IMG_SIZE = 224
CLASSES = ["fertile", "infertile", "dead"]; CLS2IDX = {c:i for i,c in enumerate(CLASSES)}
MEAN=(0.485,0.456,0.406); STD=(0.229,0.224,0.225)
TRAIN_PER_CLASS = 400   # подвыборка train (задача простая, этого хватает; уменьши для скорости)

class Letterbox:
    def __init__(s, size=IMG_SIZE): s.size=size
    def __call__(s, im):
        im=ImageOps.exif_transpose(im).convert("RGB"); w,h=im.size; k=s.size/max(w,h)
        nw,nh=max(1,round(w*k)),max(1,round(h*k)); im=im.resize((nw,nh),Image.BILINEAR)
        c=Image.new("RGB",(s.size,s.size),(0,0,0)); c.paste(im,((s.size-nw)//2,(s.size-nh)//2)); return c

tf = T.Compose([Letterbox(), T.ToTensor(), T.Normalize(MEAN,STD)])  # без аугментаций: признаки детерминированы

def cls_of(name):
    n=name.lower()
    return "infertile" if "infertile" in n else "fertile" if "fertile" in n else "dead" if "dead" in n else None

root=Path('/content/data/chicken')
allf=[p for p in root.rglob('*') if p.suffix.lower() in {'.jpg','.png','.jpeg'} and cls_of(p.name)]
train_all=[p for p in allf if 'training' in str(p).lower()]
test_files=[str(p) for p in allf if 'testing' in str(p).lower()]
# стратифицированная подвыборка train
by={c:[] for c in CLASSES}
for p in train_all: by[cls_of(p.name)].append(str(p))
train_files=[]
for c in CLASSES:
    random.shuffle(by[c]); train_files+=by[c][:TRAIN_PER_CLASS]
from collections import Counter
print("train (подвыборка):", len(train_files), Counter(cls_of(Path(p).name) for p in train_files))
print("test (полный):", len(test_files), Counter(cls_of(Path(p).name) for p in test_files))

class DS(Dataset):
    def __init__(s, files): s.files=files
    def __len__(s): return len(s.files)
    def __getitem__(s, i):
        p=s.files[i]
        with Image.open(p) as im: x=tf(im)
        return x, CLS2IDX[cls_of(Path(p).name)]


In [ ]:
# Ячейка 3 — экстрактор признаков: ResNet18 (ImageNet), fc -> Identity, только forward
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
extractor = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)  # нужен интернет (в Colab он есть)
extractor.fc = nn.Identity()                                  # выход 512-мерный эмбеддинг
extractor.eval().to(DEVICE)
for p in extractor.parameters(): p.requires_grad = False

@torch.no_grad()
def extract(files):
    dl = DataLoader(DS(files), batch_size=32, shuffle=False, num_workers=2)
    feats, ys = [], []
    for i,(x,y) in enumerate(dl):
        feats.append(extractor(x.to(DEVICE)).cpu().numpy()); ys.append(y.numpy())
        print(f"  батч {i+1}/{len(dl)}", end="\r")
    print()
    return np.concatenate(feats), np.concatenate(ys)

print("Извлекаю признаки train..."); Xtr, ytr = extract(train_files)
print("Извлекаю признаки test...");  Xte, yte = extract(test_files)
print("shapes:", Xtr.shape, Xte.shape)


In [ ]:
# Ячейка 4 — логрегрессия поверх эмбеддингов + честная оценка на test
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, recall_score
import matplotlib.pyplot as plt

scaler = StandardScaler().fit(Xtr)
clf = LogisticRegression(max_iter=2000, C=1.0).fit(scaler.transform(Xtr), ytr)
pred = clf.predict(scaler.transform(Xte))
acc = accuracy_score(yte, pred)
print("TEST accuracy:", round(acc,4))
print(classification_report(yte, pred, target_names=CLASSES, digits=3))

cm = confusion_matrix(yte, pred)
fig,ax=plt.subplots(figsize=(5,4.4)); im=ax.imshow(cm,cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(CLASSES); ax.set_yticks(range(3)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("предсказано"); ax.set_ylabel("истина"); ax.set_title(f"ResNet18 linear probe, test acc={acc:.3f}")
for i in range(3):
    for j in range(3): ax.text(j,i,cm[i,j],ha="center",va="center",color="white" if cm[i,j]>cm.max()/2 else "black")
plt.tight_layout(); plt.savefig("/content/confusion_matrix_cnn.png",dpi=120); plt.show()


In [ ]:
# Ячейка 5 — сохранить модель + метрики в Google Drive
import json, os, joblib
from sklearn.metrics import recall_score
metrics = {
  "model": "resnet18_linear_probe",
  "test_accuracy": round(float(acc),4),
  "per_class_recall": {c: round(float(r),4) for c,r in zip(CLASSES, recall_score(yte,pred,average=None))},
  "confusion_matrix": confusion_matrix(yte,pred).tolist(),
  "classes": CLASSES, "img_size": IMG_SIZE, "seed": SEED,
  "train_per_class": TRAIN_PER_CLASS, "n_test": len(test_files),
  "note": "Linear probe (замороженный ResNet18 + логрегрессия). Точность на курах ожидаемо высокая (артефакты). Реальная проверка — перепела, Этап 6."
}
OUT="/content/drive/MyDrive/verinbirds_out"; os.makedirs(OUT, exist_ok=True)
joblib.dump({"scaler":scaler, "clf":clf, "classes":CLASSES}, OUT+"/verinbirds_linear_probe.joblib")
with open(OUT+"/metrics.json","w") as f: json.dump(metrics,f,ensure_ascii=False,indent=2)
import shutil
if os.path.exists("/content/confusion_matrix_cnn.png"): shutil.copy("/content/confusion_matrix_cnn.png", OUT)
print("Сохранено в Google Drive ->", OUT)
print(json.dumps(metrics, ensure_ascii=False, indent=2))


## Готово

В **Google Drive → `verinbirds_out/`**: `verinbirds_linear_probe.joblib`, `metrics.json`, `confusion_matrix_cnn.png`.

**Сохрани напечатанный выше JSON** — для сравнения с baseline.
Модель `.joblib` положи в репозиторий `ml/models/` (она лёгкая, без весов CNN внутри — веса ResNet18
берутся из torchvision при инференсе).